In [2]:
from datasets import load_dataset

ds = load_dataset("Nan-Do/code-search-net-python")

In [3]:
ds

DatasetDict({
    train: Dataset({
        features: ['repo', 'path', 'func_name', 'original_string', 'language', 'code', 'code_tokens', 'docstring', 'docstring_tokens', 'sha', 'url', 'partition', 'summary'],
        num_rows: 455243
    })
})

In [4]:
ds['train']['original_string'][0]

'def train(train_dir, model_save_path=None, n_neighbors=None, knn_algo=\'ball_tree\', verbose=False):\n    """\n    Trains a k-nearest neighbors classifier for face recognition.\n\n    :param train_dir: directory that contains a sub-directory for each known person, with its name.\n\n     (View in source code to see train_dir example tree structure)\n\n     Structure:\n        <train_dir>/\n        ├── <person1>/\n        │   ├── <somename1>.jpeg\n        │   ├── <somename2>.jpeg\n        │   ├── ...\n        ├── <person2>/\n        │   ├── <somename1>.jpeg\n        │   └── <somename2>.jpeg\n        └── ...\n\n    :param model_save_path: (optional) path to save model on disk\n    :param n_neighbors: (optional) number of neighbors to weigh in classification. Chosen automatically if not specified\n    :param knn_algo: (optional) underlying data structure to support knn.default is ball_tree\n    :param verbose: verbosity of training\n    :return: returns knn classifier that was trained on 

In [5]:
print(repr(ds['train']["code"][0]))

'def train(train_dir, model_save_path=None, n_neighbors=None, knn_algo=\'ball_tree\', verbose=False):\n    """\n    Trains a k-nearest neighbors classifier for face recognition.\n\n    :param train_dir: directory that contains a sub-directory for each known person, with its name.\n\n     (View in source code to see train_dir example tree structure)\n\n     Structure:\n        <train_dir>/\n        ├── <person1>/\n        │   ├── <somename1>.jpeg\n        │   ├── <somename2>.jpeg\n        │   ├── ...\n        ├── <person2>/\n        │   ├── <somename1>.jpeg\n        │   └── <somename2>.jpeg\n        └── ...\n\n    :param model_save_path: (optional) path to save model on disk\n    :param n_neighbors: (optional) number of neighbors to weigh in classification. Chosen automatically if not specified\n    :param knn_algo: (optional) underlying data structure to support knn.default is ball_tree\n    :param verbose: verbosity of training\n    :return: returns knn classifier that was trained on 

In [6]:
code_strings=[]

for code in ds['train']['code']:
  code_strings.append(code)

remove comments and docstring from code

In [7]:
import io
import tokenize

def tokenize_code(source):
    tokens = []

    prev_toktype = tokenize.INDENT
    try:
      for tok in tokenize.generate_tokens(io.StringIO(source).readline):
          tok_type = tok.type
          tok_str = tok.string

          # Ignore comments
          if tok_type == tokenize.COMMENT:
              continue

          # Ignore docstrings
          if (
              tok_type == tokenize.STRING
              and prev_toktype in (
                  tokenize.INDENT,
                  tokenize.NEWLINE,
                  tokenize.DEDENT,
              )
          ):
              prev_toktype = tok_type
              continue

          if tok_type == tokenize.INDENT:
            tokens.append("<INDENT>")
          elif tok_type == tokenize.DEDENT:
            tokens.append("<DEDENT>")
          elif tok_type in (tokenize.NEWLINE, tokenize.NL):
            tokens.append("<NEWLINE>")
          else:
            tokens.append(tok_str)
          prev_toktype = tok_type


      return tokens

    except (IndentationError, tokenize.TokenError, TabError):
        return None

In [8]:
code_tokens=[]

for code in code_strings:
  token=tokenize_code(code)
  if token:
    code_tokens.append(token)

In [9]:
from sklearn.model_selection import train_test_split

train - test - valid split - 80-10-10

In [11]:
train_data,temp_data=train_test_split(code_tokens,test_size=0.2,random_state=42)
test_data,valid_data=train_test_split(temp_data,test_size=0.5,random_state=42)

building vocabulary with train data

In [12]:
vocab={}
from collections import Counter
vocab['<PAD>']=0
vocab['<UNK>']=1
vocab['<BOS>']=2
vocab['<EOS>']=3

counter=Counter()
for token_list in train_data:
  counter.update(token for token in token_list)

for token,freq in counter.most_common(50000):
  if freq>=2:
    vocab[token]=len(vocab)

In [13]:
import pickle

In [14]:
with open('vocab.pkl','wb') as f:
  pickle.dump(vocab,f)

encode tokens

In [15]:
def encode(token_list):
  return [vocab[token] if token in vocab else vocab['<UNK>'] for token in token_list]

In [24]:
max_func_len=1024

In [25]:
encoded_train = [encode(token) for token in train_data if len(token)<=max_func_len]
encoded_test = [encode(token) for token in test_data if len(token)<=max_func_len]
encoded_valid = [encode(token) for token in valid_data if len(token)<=max_func_len]

In [26]:
print(len(encoded_train))
print(len(encoded_test))
print(len(encoded_valid))

361633
45225
45216


In [27]:
lengths = [len(row) for row in encoded_train]

print("Average:", sum(lengths) / len(lengths))
print("Max:", max(lengths))
print("Min:", min(lengths))

Average: 134.20826362638365
Max: 1024
Min: 25


In [19]:
import pickle
with open('encoded_train.pkl','wb') as f:
  pickle.dump(encoded_train,f)

with open('encoded_test.pkl','wb') as f:
  pickle.dump(encoded_test,f)

with open('encoded_valid.pkl','wb') as f:
  pickle.dump(encoded_valid,f)